# NB01b – Combustão, híbrido ou elétrico: qual custa menos para rodar em cada estado?
**Métodos e Ferramentas de Data Science – FGV MBA · Aula 1 · Exercício em sala**

## A pergunta de negócio
A empresa do NB01 vai renovar a frota e está em dúvida entre três SUVs do mesmo porte: um a **combustão** (flex), um **híbrido** (flex) e um **elétrico**. O gerente de frota pergunta:

> **Em cada estado, qual tecnologia tem o menor custo anual para rodar: energia mais IPVA?**

## Premissas do exercício
- Cada carro roda **15.000 km por ano**.
- Olhamos o **primeiro ano** de um carro **0 km** comprado na capital do estado.
- Carros flex abastecem com o combustível **mais barato por km** (etanol ou gasolina), a mesma ideia da regra dos 70% do NB01.
- O elétrico é carregado na tomada de casa, pela tarifa residencial.

## Os dados (todos públicos)
| Tabela | O que traz | Fonte |
|---|---|---|
| `anp_combustiveis_2026s1.csv.gz` | Preço do etanol e da gasolina nos postos, 1º sem. 2026 (a mesma do NB01) | [ANP – Série histórica de preços](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis) |
| `aneel_tarifa_residencial_2026.csv` | Tarifa residencial de energia (TE + TUSD, **sem tributos**) da distribuidora da capital, vigente em 13/10/2026 | [ANEEL – Tarifas homologadas das distribuidoras](https://dadosabertos.aneel.gov.br/dataset/tarifas-distribuidoras-energia-eletrica) |
| `inmetro_pbev_2026_suv.csv` | Consumo dos três carros e valor FIPE de outubro/2026 | Inmetro, Tabela PBEV 2026 (atualização de 27/02/2026); FIPE |
| `ipva_2026_uf.csv` | Alíquota de IPVA 2026 do 1º ano, por estado e tecnologia | Legislação estadual compilada por fontes secundárias (ver a coluna `observacao` e a seção de limitações) |

## Como usar
Execute as células **de cima para baixo**, com **Shift + Enter**. Antes de cada célula de código há um bloco **"O que acontece na célula abaixo"**: leia primeiro, execute depois.

O exercício tem **duas lacunas** marcadas com `SEU CÓDIGO AQUI`. O professor discute as respostas com a turma ao fim do exercício.

## 1. Ferramentas

**O que acontece na célula abaixo**

As mesmas ferramentas do NB01: `pandas` para tabelas e `matplotlib` para gráficos. O `set_option` mostra os números com 2 casas decimais.

**Resultado esperado:** nenhuma saída.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:.2f}".format)

## 2. Carregar os dados
Agora são **quatro tabelas**, não uma.

**O que acontece na célula abaixo**

- `BASE` guarda o começo do endereço, comum aos quatro arquivos. Assim cada `read_csv` só completa com o nome do arquivo (`BASE + "nome.csv"` junta os dois textos).
- Cada `pd.read_csv` cria uma tabela: `anp` (preços dos combustíveis), `tarifa` (energia elétrica), `carros` (os três SUVs) e `ipva` (alíquotas por estado).
- O `print` final mostra o tamanho de cada tabela, no formato (linhas, colunas).

**Resultado esperado:** `anp` com 422.418 linhas, `tarifa` e `ipva` com 27 linhas (uma por UF) e `carros` com 3 linhas.

In [ ]:
BASE = "https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/"

anp = pd.read_csv(BASE + "anp_combustiveis_2026s1.csv.gz")
tarifa = pd.read_csv(BASE + "aneel_tarifa_residencial_2026.csv")
carros = pd.read_csv(BASE + "inmetro_pbev_2026_suv.csv")
ipva = pd.read_csv(BASE + "ipva_2026_uf.csv")

print("anp:", anp.shape, "| tarifa:", tarifa.shape, "| carros:", carros.shape, "| ipva:", ipva.shape)

## 3. Primeira olhada: os três carros

**O que acontece na célula abaixo**

- `carros` na última linha mostra a tabela inteira (são só 3 linhas).
- Colunas de consumo: `etanol_cidade`, `etanol_estrada`, `gasolina_cidade` e `gasolina_estrada` estão em **km por litro**. O elétrico não usa combustível, por isso aparece `NaN` (vazio) nessas colunas.
- `mj_km` é o **consumo de energia** em megajoules por km, que a etiqueta do Inmetro informa para todos os carros. Vamos usá-lo para o elétrico.
- `valor_fipe` é o preço de referência do carro, que serve de base para o IPVA.

**Resultado esperado:** o híbrido faz mais km por litro **na cidade** do que na estrada, ao contrário do carro a combustão. É o motor elétrico ajudando no anda e para.

In [ ]:
carros

**O que acontece na célula abaixo**

- `tarifa.head()` mostra os 5 primeiros estados da tabela de energia: a sigla da distribuidora que atende a capital e a tarifa em **R$ por kWh, sem tributos**.
- `ipva.head()` mostra as alíquotas de IPVA (em %) para cada tecnologia, com uma coluna de observação.
- Como são duas tabelas, usamos `display(...)` para mostrar as duas na mesma célula (sem ele, só a última apareceria).

**Resultado esperado:** duas tabelas com AC, AL, AM, AP e BA. Repare que, em vários estados, o elétrico e o híbrido têm alíquota 0.

In [ ]:
display(tarifa.head())
display(ipva.head())

## 4. Preço dos combustíveis por estado
É o mesmo passo do NB01.

**O que acontece na célula abaixo**

- Filtramos a gasolina comum e o etanol com `isin`, como no NB01.
- O `groupby(["uf", "produto"])` calcula o preço médio de cada combustível em cada estado; o `.unstack()` deixa os produtos como colunas.

**Resultado esperado:** 27 linhas (estados) e 2 colunas, `ETANOL` e `GASOLINA`, em R$ por litro.

In [ ]:
flex = anp[anp["produto"].isin(["GASOLINA", "ETANOL"])]
precos = flex.groupby(["uf", "produto"])["preco_venda"].mean().unstack()
precos.head()

## 5. Quanto cada carro gasta por km
Para cada carro: **custo por km = preço da energia ÷ quantos km ela rende**.

**O que acontece na célula abaixo**

A etiqueta traz o consumo na cidade e na estrada. Juntamos os dois num **consumo combinado**, com o peso usual de **55% cidade e 45% estrada**.

- `carros = carros.set_index("tecnologia")` faz a coluna `tecnologia` virar o **índice** da tabela. Assim, depois, buscamos um carro pelo nome: `carros.loc["Híbrido", ...]`.
- As duas linhas seguintes criam as colunas `etanol_kml` e `gasolina_kml`: 0,55 × cidade + 0,45 × estrada. A conta é feita para os três carros de uma vez.
- A última linha mostra só as colunas que interessam, com a lista de nomes entre colchetes duplos.

**Resultado esperado:** combustão com cerca de 8,5 km/l no etanol e 12,2 km/l na gasolina; híbrido com cerca de 10,8 e 15,4. O elétrico fica `NaN`.

In [ ]:
carros = carros.set_index("tecnologia")
carros["etanol_kml"] = 0.55 * carros["etanol_cidade"] + 0.45 * carros["etanol_estrada"]
carros["gasolina_kml"] = 0.55 * carros["gasolina_cidade"] + 0.45 * carros["gasolina_estrada"]
carros[["modelo", "etanol_kml", "gasolina_kml", "mj_km", "valor_fipe"]]

**O que acontece na célula abaixo**

Agora o custo por km dos dois carros flex, em cada estado.

- `custo_km = pd.DataFrame(index=precos.index)` cria uma tabela vazia, com uma linha por estado (o mesmo índice da tabela `precos`).
- O `for tec in ["Combustão", "Híbrido"]:` repete o bloco de dentro duas vezes: uma para cada carro flex. O bloco de dentro é o que está **recuado** (com espaços no começo).
- `km_etanol = carros.loc[tec, "etanol_kml"]` busca quantos km o carro faz com um litro de etanol.
- `precos["ETANOL"] / km_etanol` dá o **custo por km com etanol** em cada estado (R$ por litro ÷ km por litro = R$ por km). O mesmo para a gasolina.
- `pd.concat([...], axis=1)` põe as duas colunas lado a lado; `.min(axis=1)` escolhe, **em cada linha**, o menor valor. É o motorista escolhendo o combustível mais barato por km.

**Resultado esperado:** valores entre R$ 0,40 e R$ 0,65 por km. O híbrido sai cerca de 20% mais barato que o carro a combustão.

In [ ]:
custo_km = pd.DataFrame(index=precos.index)

for tec in ["Combustão", "Híbrido"]:
    km_etanol = carros.loc[tec, "etanol_kml"]
    km_gasolina = carros.loc[tec, "gasolina_kml"]
    com_etanol = precos["ETANOL"] / km_etanol
    com_gasolina = precos["GASOLINA"] / km_gasolina
    custo_km[tec] = pd.concat([com_etanol, com_gasolina], axis=1).min(axis=1)

custo_km.head()

**O que acontece na célula abaixo**

E o elétrico, que gasta energia elétrica, não litros.

- A etiqueta dá o consumo em **MJ/km**. Como 1 kWh = 3,6 MJ, dividir por 3,6 converte para **kWh por km**.
- `kwh_km × tarifa` dá o custo por km: kWh por km × R$ por kWh = R$ por km.
- `FATOR_TRIBUTOS = 1.0` usa a tarifa **sem tributos**, como vem da ANEEL. Na seção 9 você vai mudar esse número para ver o efeito do ICMS e do PIS/Cofins.
- `tarifa.set_index("uf")["tarifa_kwh_sem_tributos"]` pega a coluna de tarifa com os estados como índice, para casar linha a linha com a tabela `custo_km`.

**Resultado esperado:** o elétrico custa cerca de R$ 0,11 a R$ 0,16 por km, perto de um quarto do custo do carro a combustão.

In [ ]:
FATOR_TRIBUTOS = 1.0   # 1.0 = tarifa sem tributos; veja a seção 9

kwh_km = carros.loc["Elétrico", "mj_km"] / 3.6
tarifa_uf = tarifa.set_index("uf")["tarifa_kwh_sem_tributos"] * FATOR_TRIBUTOS
custo_km["Elétrico"] = kwh_km * tarifa_uf

custo_km.head()

## 6. Custo anual: energia + IPVA

**O que acontece na célula abaixo**

- `energia = custo_km * KM_POR_ANO` multiplica toda a tabela por 15.000: o custo de energia de um ano, por estado e tecnologia.
- `ipva_uf = ipva.set_index("uf")` deixa os estados como índice da tabela de IPVA.
- A tabela `imposto` calcula o IPVA de cada carro em cada estado: **alíquota ÷ 100 × valor FIPE do carro**. O dicionário `{"Combustão": ..., "Híbrido": ..., "Elétrico": ...}` dá o nome de cada coluna.
- `anual = energia + imposto` soma as duas tabelas, célula a célula (mesmo estado, mesma tecnologia).
- `anual["mais_barato"] = anual.idxmin(axis=1)` procura, em cada linha, **o nome da coluna** com o menor valor: a tecnologia vencedora no estado.

**Resultado esperado:** uma tabela de 27 estados com o custo anual das três tecnologias e a vencedora de cada um.

In [ ]:
KM_POR_ANO = 15000

energia = custo_km * KM_POR_ANO

ipva_uf = ipva.set_index("uf")
imposto = pd.DataFrame({
    "Combustão": ipva_uf["aliquota_combustao"] / 100 * carros.loc["Combustão", "valor_fipe"],
    "Híbrido": ipva_uf["aliquota_hibrido"] / 100 * carros.loc["Híbrido", "valor_fipe"],
    "Elétrico": ipva_uf["aliquota_eletrico"] / 100 * carros.loc["Elétrico", "valor_fipe"],
})

anual = energia + imposto
anual["mais_barato"] = anual.idxmin(axis=1)
anual.sort_values("Elétrico")

**O que acontece na célula abaixo**

- `anual["mais_barato"].value_counts()` conta em quantos estados cada tecnologia é a mais barata.

**Resultado esperado:** uma tecnologia ganha em quase todos os estados, mas não em todos.

In [ ]:
anual["mais_barato"].value_counts()

**Pare e pense:** em qual estado o resultado é diferente? Olhe a tabela `imposto` desse estado e explique por quê.

## 7. O gráfico

**O que acontece na célula abaixo**

Esta célula tem uma **lacuna**: troque `____` pelo nome da tabela com o custo anual (sem a coluna de texto).

- `[["Combustão", "Híbrido", "Elétrico"]]` seleciona só as três colunas de números; a coluna `mais_barato` é texto e não entra no gráfico.
- `.sort_values("Elétrico")` ordena os estados pelo custo do elétrico.
- `.plot(kind="bar", ...)` desenha um gráfico de barras agrupadas: três barras por estado, uma por tecnologia.

**Resultado esperado:** em quase todos os estados, a barra do elétrico é a menor.

In [ ]:
# SEU CÓDIGO AQUI: troque ____ pelo nome da tabela com o custo anual
____[["Combustão", "Híbrido", "Elétrico"]].sort_values("Elétrico").plot(kind="bar", figsize=(14, 4))
plt.title("Custo anual de energia + IPVA no 1º ano, por estado (15.000 km/ano)")
plt.ylabel("R$ por ano")
plt.show()

## 8. Seu estado

**O que acontece na célula abaixo**

Esta célula tem uma **lacuna**: troque `"__"` pela sigla do estado, entre aspas e em maiúsculas.

- `pd.DataFrame({...})` monta uma tabela pequena só com o estado escolhido: energia, IPVA e total de cada tecnologia. O `.loc[ESTADO]` busca a linha do estado em cada tabela.
- `ipva_uf.loc[ESTADO, "observacao"]` mostra a observação sobre o IPVA daquele estado (por exemplo, se a regra precisa ser conferida).

**Resultado esperado:** uma tabela de 3 linhas (as tecnologias) e 3 colunas (energia, IPVA e total), a tecnologia mais barata e a observação do IPVA.

In [ ]:
ESTADO = "__"   # SEU CÓDIGO AQUI: troque pela sigla do estado, por exemplo "RJ"

resumo = pd.DataFrame({
    "energia": energia.loc[ESTADO],
    "ipva": imposto.loc[ESTADO],
    "total": anual.loc[ESTADO, ["Combustão", "Híbrido", "Elétrico"]],
})
display(resumo)
print("Mais barato em", ESTADO, "→", anual.loc[ESTADO, "mais_barato"])
print("Observação do IPVA:", ipva_uf.loc[ESTADO, "observacao"])

## 9. E se a energia elétrica tiver tributos?
A tarifa da ANEEL vem **sem tributos**, mas o preço da bomba já inclui os impostos. Comparar os dois direto **favorece o elétrico**.

**Teste:** volte à seção 5, troque `FATOR_TRIBUTOS = 1.0` por `FATOR_TRIBUTOS = 1.3` (30% a mais, um valor hipotético para o efeito do ICMS e do PIS/Cofins, que variam por estado) e execute de novo dali para baixo (*Ambiente de execução → Executar a partir daqui*).

**Pergunta:** o vencedor muda em algum estado?

## 10. Antes de levar à diretoria: limitações
1. **Só o primeiro ano.** Várias isenções de IPVA valem só no 1º ano ou só até 2026. Nos anos seguintes, a conta muda.
2. **Preço de compra fora da conta.** O elétrico e o híbrido custam mais que o carro a combustão. A pergunta certa para a diretoria é em quanto tempo a economia paga essa diferença.
3. **Tarifa sem tributos** e de uma só distribuidora por estado (a da capital). O interior pode ter outra distribuidora e outra tarifa.
4. **IPVA de fontes secundárias.** A tabela foi compilada de reportagens e guias que nem sempre concordam. Os estados marcados com "conferir" precisam ser checados na Secretaria da Fazenda antes de qualquer decisão.
5. **Consumo de laboratório.** A etiqueta mede em condições padrão. O consumo real depende do motorista, do trânsito e, no elétrico, da recarga (em casa ou em eletroposto, que custa mais).
6. **Fora da conta:** manutenção, seguro, revenda e a infraestrutura de recarga da frota.

> O dado responde à pergunta que você fez. Se a pergunta deixa custos de fora, a resposta também deixa.

## Erros comuns e como resolver

| Mensagem | O que aconteceu | O que fazer |
|---|---|---|
| `NameError: name 'pd' is not defined` | Você pulou a célula do `import` | *Ambiente de execução → Executar anteriores* |
| `KeyError: 'Híbrido'` na seção 5 | A célula do `set_index("tecnologia")` rodou duas vezes ou não rodou | Execute de novo desde a seção 2 |
| `KeyError: '__'` | Sigla do estado não preenchida ou errada | Use 2 letras maiúsculas, entre aspas: `"RJ"` |
| `NameError: name '____' is not defined` | A lacuna do gráfico não foi preenchida | Troque `____` pelo nome da tabela de custo anual |
| `HTTPError` ou `URLError` na seção 2 | Sem internet ou rede bloqueada | Baixe os 4 arquivos do ECLASS, arraste para a pasta do Colab e troque `BASE` por `""` |

### Autor e licença

**Autor:** Prof. Carlos Carlim · [LinkedIn](https://www.linkedin.com/in/carloscarlim-77144120) · [GitHub](https://github.com/carloscarlim)

**Licença MIT.** Copyright (c) 2026 Carlos Carlim. Os direitos autorais deste material pertencem exclusivamente ao autor.

É permitido usar, copiar, modificar, mesclar, publicar, distribuir, sublicenciar e vender cópias deste material, desde que este aviso de copyright e o texto da licença sejam mantidos em todas as cópias ou partes substanciais. O material é fornecido "no estado em que se encontra", sem garantia de qualquer tipo. Texto completo da licença: [LICENSE](https://github.com/carloscarlim/fgv-mfds/blob/main/LICENSE).

Os dados usados nos notebooks têm licenças próprias, indicadas em [`dados/README.md`](https://github.com/carloscarlim/fgv-mfds/blob/main/dados/README.md).